# Feature Selection Pipeline

## Overview

This notebook performs feature selection for the multimodal breast cancer recurrence prediction dataset.

The objective is to reduce genomic dimensionality while preserving clinically informative features and avoiding data leakage.

Feature selection was performed using only the training dataset and then applied to validation and test datasets.

## Pipeline

1. Load train, validation, and test datasets.
2. Separate clinical and genomic modalities.
3. Remove constant genomic features using Variance Threshold.
4. Remove highly correlated genomic features (correlation > 0.95).
5. Select the top 200 genes using ANOVA F-test based on recurrence labels.
6. Apply the selected feature set to validation and test datasets.
7. Save the final selected datasets and selected gene list.

## Results

Initial genomic features:
- 1000 genes

After variance filtering:
- 1000 genes

After correlation filtering:
- 994 genes

Final selected genomic features:
- 200 genes

Final dataset size:

Training:
- 249 patients
- 224 features

Validation:
- 54 patients
- 224 features

Testing:
- 54 patients
- 224 features

The final datasets contain:
- Clinical features
- Selected genomic features
- Histopathology identifiers
- Survival outcome labels

All splits were validated to ensure:
- No duplicated patients
- No feature mismatch
- No missing values

The generated datasets are ready for model development.

##Imports

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_classif
)

from sklearn.preprocessing import StandardScaler


import warnings
warnings.filterwarnings("ignore")


print("Libraries imported successfully.")

Libraries imported successfully.


##Mount Google Drive

In [2]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


##Define Paths

In [3]:
# ============================================================
# PROJECT PATHS
# ============================================================


BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"


DATASET_DIR = os.path.join(
    BASE_DIR,
    "dataset_split"
)


SAVE_DIR = os.path.join(
    BASE_DIR,
    "feature_selection"
)


os.makedirs(
    SAVE_DIR,
    exist_ok=True
)



TRAIN_PATH = os.path.join(
    DATASET_DIR,
    "train.csv"
)


VAL_PATH = os.path.join(
    DATASET_DIR,
    "validation.csv"
)


TEST_PATH = os.path.join(
    DATASET_DIR,
    "test.csv"
)



print("Paths configured.")

Paths configured.


##Load Dataset Splits

In [4]:
# ============================================================
# LOAD DATASETS
# ============================================================


train = pd.read_csv(TRAIN_PATH)

validation = pd.read_csv(VAL_PATH)

test = pd.read_csv(TEST_PATH)



print("="*70)
print("DATASETS LOADED")
print("="*70)


print("Train:")
print(train.shape)


print("\nValidation:")
print(validation.shape)


print("\nTest:")
print(test.shape)

DATASETS LOADED
Train:
(249, 1024)

Validation:
(54, 1024)

Test:
(54, 1024)


##Dataset Quality Control

In [5]:
# ============================================================
# QUALITY CONTROL
# ============================================================


datasets = {
    "Train": train,
    "Validation": validation,
    "Test": test
}



for name, df in datasets.items():

    print("="*70)
    print(name)

    print(
        "Patients:",
        df["patient_id"].nunique()
    )

    print(
        "Duplicates:",
        df["patient_id"].duplicated().sum()
    )

    print(
        "Missing values:",
        df.isna().sum().sum()
    )

    print(
        "Events:",
        df["event"].sum()
    )

    print()

Train
Patients: 249
Duplicates: 0
Missing values: 0
Events: 63

Validation
Patients: 54
Duplicates: 0
Missing values: 0
Events: 14

Test
Patients: 54
Duplicates: 0
Missing values: 0
Events: 14



##Separate Feature Groups

In [12]:
# ============================================================
# FEATURE GROUP IDENTIFICATION
# ============================================================


print("="*70)
print("IDENTIFYING FEATURE GROUPS")
print("="*70)



# Fixed columns

non_feature_columns = [
    "patient_id",
    "file_uuid",
    "event",
    "survival_time"
]



# Clinical features
clinical_features = [
    "years_to_birth",
    "Tumor_purity",
    "pathologic_stage",
    "pathology_T_stage",
    "pathology_N_stage",
    "pathology_M_stage",
    "number_of_lymph_nodes",
    "radiation_therapy"
]


clinical_features += [
    c for c in train.columns
    if c.startswith(
        (
            "histological_type",
            "PAM50",
            "race",
            "ethnicity"
        )
    )
]



# Genomic features

genomic_features = [
    c for c in train.columns
    if c not in (
        non_feature_columns
        +
        clinical_features
    )
]



print(
    "Clinical features:",
    len(clinical_features)
)


print(
    "Genomic features:",
    len(genomic_features)
)


print(
    "First genomic features:"
)


print(
    genomic_features[:20]
)

IDENTIFYING FEATURE GROUPS
Clinical features: 20
Genomic features: 1000
First genomic features:
['CLEC3A', 'CPB1', 'SCGB2A2', 'SCGB1D2', 'TFF1', 'GSTM1', 'PIP', 'S100A7', 'MUCL1', 'CYP2B7P1', 'ANKRD30A', 'PRAME', 'CYP4Z1', 'KCNJ3', 'AGR3', 'HMGCS2', 'SERPINA6', 'TFAP2B', 'MUC6', 'DHRS2']


##Debugging cells

In [13]:
# ============================================================
# DEBUG GENOMIC FEATURES
# ============================================================

print("="*70)
print("GENOMIC FEATURE DEBUG")
print("="*70)


print("Number of genomic features:")
print(len(genomic_features))


print("\nFirst genomic features:")
print(genomic_features[:10])


print("\nTrain dataset shape:")
print(train.shape)


print("\nColumns containing genomic-like data:")

gene_like = [
    c for c in train.columns
    if c not in [
        "patient_id",
        "event",
        "survival_time",
        "file_uuid"
    ]
]

print(len(gene_like))

print(gene_like[:20])

GENOMIC FEATURE DEBUG
Number of genomic features:
1000

First genomic features:
['CLEC3A', 'CPB1', 'SCGB2A2', 'SCGB1D2', 'TFF1', 'GSTM1', 'PIP', 'S100A7', 'MUCL1', 'CYP2B7P1']

Train dataset shape:
(249, 1024)

Columns containing genomic-like data:
1020
['years_to_birth', 'Tumor_purity', 'pathologic_stage', 'pathology_T_stage', 'pathology_N_stage', 'pathology_M_stage', 'number_of_lymph_nodes', 'radiation_therapy', 'histological_type_infiltratinglobularcarcinoma', 'histological_type_medullarycarcinoma', 'histological_type_metaplasticcarcinoma', 'histological_type_mixedhistology(pleasespecify)', 'histological_type_mucinouscarcinoma', 'histological_type_other,specify', 'PAM50_Her2', 'PAM50_LumA', 'PAM50_LumB', 'race_blackorafricanamerican', 'race_white', 'ethnicity_nothispanicorlatino']


In [9]:
# ============================================================
# INSPECT MASTER DATASET COLUMNS
# ============================================================

print("="*70)
print("MASTER DATASET COLUMNS")
print("="*70)


for i, col in enumerate(train.columns):

    print(i, col)

    if i > 50:
        break

MASTER DATASET COLUMNS
0 patient_id
1 file_uuid
2 years_to_birth
3 Tumor_purity
4 pathologic_stage
5 pathology_T_stage
6 pathology_N_stage
7 pathology_M_stage
8 number_of_lymph_nodes
9 radiation_therapy
10 histological_type_infiltratinglobularcarcinoma
11 histological_type_medullarycarcinoma
12 histological_type_metaplasticcarcinoma
13 histological_type_mixedhistology(pleasespecify)
14 histological_type_mucinouscarcinoma
15 histological_type_other,specify
16 PAM50_Her2
17 PAM50_LumA
18 PAM50_LumB
19 race_blackorafricanamerican
20 race_white
21 ethnicity_nothispanicorlatino
22 CLEC3A
23 CPB1
24 SCGB2A2
25 SCGB1D2
26 TFF1
27 GSTM1
28 PIP
29 S100A7
30 MUCL1
31 CYP2B7P1
32 ANKRD30A
33 PRAME
34 CYP4Z1
35 KCNJ3
36 AGR3
37 HMGCS2
38 SERPINA6
39 TFAP2B
40 MUC6
41 DHRS2
42 SLC30A8
43 UGT2B11
44 VSTM2A
45 COL2A1
46 C4orf7
47 TAT
48 ADIPOQ
49 ADH1B
50 CALML5
51 GP2


In [10]:
# ============================================================
# CHECK LAST COLUMNS
# ============================================================

print("="*70)
print("LAST COLUMNS")
print("="*70)


for i, col in enumerate(train.columns[-50:]):

    print(col)

LAST COLUMNS
FXYD1
CES1
CDH7
VIT
PNPLA3
LEMD1
DPP10
HDC
IL6
CACNA2D1
ZPLD1
CCDC129
LDHC
LOC284379
KLHL1
C20orf85
CHRNB2
MMRN1
EDIL3
PEX5L
MLC1
psiTPTE22
BMP5
MAGEC2
CXCL11
PLA2G10
TGM5
CD207
COL11A2
SNAP25
CDCA7
CLDN10
OBP2A
ENHO
EPHA7
B3GAT1
COL10A1
CRLF1
DNAH5
DMKN
SLC1A6
PTGDS
TFCP2L1
ZIC5
CRYAB
EMILIN3
TNNT2
INA
event
survival_time


##Remove Constant Genes

In [14]:
# ============================================================
# REMOVE CONSTANT GENOMIC FEATURES
# ============================================================


X_train_genes = train[genomic_features]


variance_selector = VarianceThreshold(
    threshold=0
)


X_train_variance = variance_selector.fit_transform(
    X_train_genes
)



selected_after_variance = (
    X_train_genes.columns[
        variance_selector.get_support()
    ]
)



print("="*70)
print("VARIANCE FILTER")
print("="*70)


print(
    "Original genes:",
    len(genomic_features)
)


print(
    "Remaining genes:",
    len(selected_after_variance)
)

VARIANCE FILTER
Original genes: 1000
Remaining genes: 1000


##Remove Highly Correlated Genes

In [15]:
# ============================================================
# REMOVE HIGHLY CORRELATED GENES
# ============================================================


corr_matrix = (
    train[selected_after_variance]
    .corr()
)



upper_triangle = (
    corr_matrix
    .where(
        np.triu(
            np.ones(
                corr_matrix.shape
            ),
            k=1
        ).astype(bool)
    )
)



correlated_features = [
    column
    for column in upper_triangle.columns
    if any(
        upper_triangle[column] > 0.95
    )
]



genes_after_corr = [
    g for g in selected_after_variance
    if g not in correlated_features
]



print("="*70)
print("CORRELATION FILTER")
print("="*70)


print(
    "Removed:",
    len(correlated_features)
)


print(
    "Remaining:",
    len(genes_after_corr)
)

CORRELATION FILTER
Removed: 6
Remaining: 994


##ANOVA Feature Selection

In [16]:
# ============================================================
# ANOVA FEATURE SELECTION
# ============================================================


TOP_K = 200



selector = SelectKBest(
    score_func=f_classif,
    k=TOP_K
)



X_train_final = selector.fit_transform(
    train[genes_after_corr],
    train["event"]
)



selected_genes = (
    pd.Series(
        genes_after_corr
    )
    [
        selector.get_support()
    ]
    .tolist()
)



print("="*70)
print("ANOVA SELECTION")
print("="*70)


print(
    "Selected genes:",
    len(selected_genes)
)

ANOVA SELECTION
Selected genes: 200


##Apply Selection to All Splits

In [17]:
# ============================================================
# APPLY FEATURE SELECTION
# ============================================================


common_columns = (
    clinical_features
    +
    selected_genes
    +
    pathology_features
    +
    [
        "patient_id",
        "event",
        "survival_time"
    ]
)



train_selected = train[
    common_columns
]


validation_selected = validation[
    common_columns
]


test_selected = test[
    common_columns
]



print(train_selected.shape)

print(validation_selected.shape)

print(test_selected.shape)

(249, 224)
(54, 224)
(54, 224)


##Save Datasets

In [18]:
# ============================================================
# SAVE DATASETS
# ============================================================


train_selected.to_csv(
    os.path.join(
        SAVE_DIR,
        "train_selected.csv"
    ),
    index=False
)



validation_selected.to_csv(
    os.path.join(
        SAVE_DIR,
        "validation_selected.csv"
    ),
    index=False
)



test_selected.to_csv(
    os.path.join(
        SAVE_DIR,
        "test_selected.csv"
    ),
    index=False
)



print("Datasets saved.")

Datasets saved.


##Save Selected Gene List

In [19]:
# ============================================================
# SAVE SELECTED GENES
# ============================================================


pd.DataFrame(
    {
        "selected_genes": selected_genes
    }
).to_csv(
    os.path.join(
        SAVE_DIR,
        "selected_genes.csv"
    ),
    index=False
)



print(
    "Gene list saved."
)

Gene list saved.


##Feature Selection Report

In [20]:
# ============================================================
# FEATURE SELECTION REPORT
# ============================================================


report = f"""

FEATURE SELECTION REPORT
========================

Original genomic features:
{len(genomic_features)}

After variance filtering:
{len(selected_after_variance)}

Removed correlated features:
{len(correlated_features)}

Final selected genes:
{len(selected_genes)}

Selection method:
Variance Threshold
Correlation filtering
ANOVA F-test

Top K:
{TOP_K}


Training patients:
{len(train)}

Validation patients:
{len(validation)}

Test patients:
{len(test)}

"""


with open(
    os.path.join(
        SAVE_DIR,
        "feature_selection_report.txt"
    ),
    "w"
) as f:

    f.write(report)



print(report)



FEATURE SELECTION REPORT

Original genomic features:
1000

After variance filtering:
1000

Removed correlated features:
6

Final selected genes:
200

Selection method:
Variance Threshold
Correlation filtering
ANOVA F-test

Top K:
200


Training patients:
249

Validation patients:
54

Test patients:
54




##Final Validation

In [21]:
# ============================================================
# FINAL VALIDATION
# ============================================================


print("="*70)
print("FINAL VALIDATION")
print("="*70)



print(
    "Train:",
    train_selected.shape
)


print(
    "Validation:",
    validation_selected.shape
)


print(
    "Test:",
    test_selected.shape
)



print(
    "\nSame columns:"
)


print(
    train_selected.columns.equals(
        validation_selected.columns
    )
)


print(
    validation_selected.columns.equals(
        test_selected.columns
    )
)



print(
    "\nMissing values:"
)


print(
    train_selected.isna().sum().sum()
)


print(
    validation_selected.isna().sum().sum()
)


print(
    test_selected.isna().sum().sum()
)

FINAL VALIDATION
Train: (249, 224)
Validation: (54, 224)
Test: (54, 224)

Same columns:
True
True

Missing values:
0
0
0
